# ProteinLoop AMD Gemma Verifier-Feedback Repair

This notebook reproduces the AMD-hosted Gemma evidence used by ProteinLoop. Gemma proposes and repairs structured recovery actions; deterministic ecosystem rules remain the only mutation authority. It records actions, verifier feedback, product outcomes, runtime versions, and API usage metadata, never private chain-of-thought or credentials.

In [ ]:
from pathlib import Path
import json
import os
import subprocess

REPO = Path(os.environ.get("PROTEINLOOP_NOTEBOOK_REPO", "/workspace/ProteinLoop"))
ENDPOINT = os.environ.get("GEMMA_ENDPOINT", "http://127.0.0.1:8001")
MODEL = os.environ.get("GEMMA_MODEL", "google/gemma-4-E2B-it")
PYTHON = os.environ.get("AMD_NOTEBOOK_PYTHON", "/workspace/proteinloop-amd/vllm-gemma4/bin/python")
BUNDLE = Path("/workspace/proteinloop-amd-roundtrip.zip")
assert REPO.exists(), f"Repository not found: {REPO}"
os.chdir(REPO)
print("Repository:", REPO)
print("Endpoint:", ENDPOINT)
print("Model:", MODEL)

## 1. Optional fresh vLLM setup

Set `RUN_SETUP` to `True` only when the endpoint is not already serving Gemma. The script reuses a cached model and environment when available. On a first gated download it prompts securely for a Hugging Face read token. Initial ROCm compilation can take 25-35 minutes.

In [ ]:
RUN_SETUP = False
if RUN_SETUP:
    subprocess.run(["bash", "scripts/amd_notebook_setup_vllm.sh"], check=True)
else:
    print("Setup skipped; using the existing endpoint.")

## 2. Verify the private endpoint

In [ ]:
import urllib.request

with urllib.request.urlopen(f"{ENDPOINT}/v1/models", timeout=10) as response:
    models = json.load(response)
model_ids = [item["id"] for item in models.get("data", [])]
assert MODEL in model_ids, (MODEL, model_ids)
print("Ready:", MODEL)

## 3. Run runtime proof, policy search, product audit, and verifier repair

This invokes `amd_notebook_run_all.sh`, which includes the 20-emergency verifier-feedback evaluation and builds `proteinloop-amd-roundtrip.zip`. BashUpload remains disabled by default.

In [ ]:
env = {**os.environ, "AMD_NOTEBOOK_PYTHON": PYTHON, "GEMMA_ENDPOINT": ENDPOINT, "GEMMA_MODEL": MODEL, "BASHUPLOAD": "0"}
subprocess.run(["bash", "scripts/amd_notebook_run_all.sh"], check=True, env=env)

## 4. Inspect measured contribution

In [ ]:
artifact_path = REPO / "submission" / "amd-gemma-repair-evaluation.json"
artifact = json.loads(artifact_path.read_text())
summary = artifact["summary"]
for key in [
    "scenario_count",
    "first_answer_safe_rate",
    "repair_path_safe_rate",
    "best_of_n_safe_rate",
    "combined_model_safe_rate",
    "final_system_safe_rate",
    "repair_rescue_count",
    "deterministic_fallback_count",
    "protected_aquatic_biomass_kg",
    "model_request_count",
    "observed_completion_tokens_per_second",
]:
    print(f"{key}: {summary[key]}")

## 5. Download or temporarily upload the evidence bundle

The preferred path is the Jupyter file browser: download `/workspace/proteinloop-amd-roundtrip.zip`. Set `UPLOAD_TO_BASHUPLOAD=True` only when a temporary return link is necessary. The bundle builder validates all artifacts and scans for credentials before creating the ZIP.

In [ ]:
assert BUNDLE.exists(), BUNDLE
subprocess.run(["sha256sum", str(BUNDLE)], check=True)
print("Download from Jupyter file browser:", BUNDLE)

UPLOAD_TO_BASHUPLOAD = False
if UPLOAD_TO_BASHUPLOAD:
    upload_env = {**env, "AMD_BUNDLE_PATH": str(BUNDLE), "BASHUPLOAD_EXPIRATION_SECONDS": "3600"}
    subprocess.run(["bash", "scripts/amd_notebook_upload_bundle.sh"], check=True, env=upload_env)
else:
    print("Temporary external upload disabled.")